## 1. Imports

In [1]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import tensorflow as tf

import os, random

from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.preprocessing import StandardScaler
from sklearn.svm import SVC
from tensorflow.keras.applications import DenseNet121, InceptionV3, VGG16, VGG19
from tensorflow.keras.callbacks import EarlyStopping
from tensorflow.keras.layers import Conv2D, Dense, Dropout, Flatten, Input, MaxPool2D
from tensorflow.keras.models import Sequential
from tensorflow.keras.preprocessing.image import ImageDataGenerator

## 2. Initial Setup

In [3]:
SEED = 42
tf.random.set_seed(SEED)
np.random.seed(SEED)
random.seed(SEED)

# Mixed-prisision on GPU
if tf.config.list_physical_devices('GPU'):
    tf.keras.mixed_precision.set_global_policy('mixed_float16')

def cm_table(cm, names):
    return pd.DataFrame(cm, index = names, columns = names)

def quick_report(model, ds):
    y_true = ds.classes
    y_pred = (model.predict(ds, verbose = 0) > 0.5).astype(int).ravel()
    return accuracy_score(y_true, y_pred), cm_table(confusion_matrix(y_true, y_pred), list(ds.class_indices))

INFO:tensorflow:Mixed precision compatibility check (mixed_float16): OK
Your GPU will likely run quickly with dtype policy mixed_float16 as it has compute capability of at least 7.0. Your GPU: NVIDIA GeForce RTX 4060, compute capability 8.9


## 3. Load Data

In [5]:
DATA_DIR = "homer_bart_1"
TEST_DIR = "test_dataset"

BATCH = 32

gen = ImageDataGenerator(rescale=1/255, validation_split=0.2, rotation_range=20, width_shift_range=0.2,
                        height_shift_range=0.2, zoom_range=0.2, shear_range=0.2, horizontal_flip=True)

train_gen = gen.flow_from_directory(DATA_DIR, target_size=(224,224), class_mode='binary', batch_size=BATCH,
                                    subset='training', seed=SEED)
val_gen   = gen.flow_from_directory(DATA_DIR, target_size=(224,224), class_mode='binary', batch_size=BATCH,
                                    subset='validation', seed=SEED)
test_gen  = ImageDataGenerator(rescale=1/255.).flow_from_directory(TEST_DIR, target_size=(224,224), 
                                                                   class_mode='binary', batch_size=1, shuffle=False)

print("Class map:", train_gen.class_indices)

Found 216 images belonging to 2 classes.
Found 53 images belonging to 2 classes.
Found 7 images belonging to 2 classes.
Class map: {'bart': 0, 'homer': 1}


## 4. Model Zoo